# Практическая работа № 3. Вариант 7


## Подготовка и проверка структуры
Функция `check_structure` вызывает `head()`, `info()` и `dtypes` после каждого чтения. Чтобы вывод порций не занимал много места, подробности помещены в раскрывающиеся блоки.

In [2]:
from pathlib import Path
from time import perf_counter
from io import StringIO
from html import escape
from IPython.display import display, HTML
import pandas as pd

DATA = next((p for p in [Path("data"), Path(".")]
             if (p / "products.xlsx").exists()
             and list(p.glob("orders_*.csv"))), Path("data"))
files = sorted(DATA.glob("orders_*.csv"))
assert files and (DATA / "products.xlsx").exists(), (
    "Не найдены данные. Укажите в DATA папку с orders_*.csv и products.xlsx."
)
print("Папка:", DATA)
print("CSV:", ", ".join(f.name for f in files))

def check_structure(df, title):
    preview = df.head()
    buffer = StringIO()
    df.info(buf=buffer)
    types = df.dtypes
    display(HTML(
        "<details><summary>" + escape(title) + "</summary>"
        + preview.to_html() + "<pre>" + escape(buffer.getvalue())
        + "\nТипы данных:\n" + escape(str(types)) + "</pre></details>"
    ))

Папка: data
CSV: orders_2024_01.csv, orders_2024_02.csv, orders_2024_03.csv


## 1. Чтение CSV по частям и фильтрация
Используем `chunksize=1000`, выбираем столбцы через `usecols`, задаём типы через `dtype`. Каждую порцию сразу фильтруем по точному значению `paid`. Неоплаченные строки не добавляются в итоговую таблицу. Счётчики ведём отдельно по каждому файлу.

В памяти сохраняются оплаченные позиции; для данного объёма данных этого достаточно.

In [3]:
columns = ["order_id", "user_id", "order_datetime", "product_id", "price", "quantity", "status"]
dtypes = {"order_id": "int64", "user_id": "int64", "product_id": "int64",
          "price": "float64", "quantity": "int32", "status": "string"}
parts, counts = [], []
for file in files:
    total = kept = 0
    for i, chunk in enumerate(pd.read_csv(
        file, chunksize=1000, usecols=columns, dtype=dtypes
    ), start=1):
        check_structure(chunk, f"{file.name}: порция {i}")
        selected = chunk.loc[chunk["status"].eq("paid")].copy()
        total += len(chunk)
        kept += len(selected)
        parts.append(selected)
    counts.append({"file": file.name, "total": total,
                   "paid": kept, "discarded": total - kept})

paid = pd.concat(parts, ignore_index=True)
counts = pd.DataFrame(counts)
assert paid["status"].eq("paid").all()
assert not paid[["order_id", "product_id", "price", "quantity"]].isna().any().any()
display(counts)

,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid


,order_id,user_id,order_datetime,product_id,price,quantity,status
1000,1001,1062,2024-01-27 04:38:30,4,12633.16,2,paid
1001,1002,1041,2024-01-28 00:10:28,27,11952.31,4,paid
1002,1003,1074,2024-01-21 12:05:28,10,2470.16,1,paid
1003,1004,1015,2024-01-31 11:11:26,19,9040.29,3,paid
1004,1005,1006,2024-01-07 10:27:40,9,18330.34,4,paid


,order_id,user_id,order_datetime,product_id,price,quantity,status
2000,2001,1073,2024-01-06 19:34:02,27,19380.76,4,paid
2001,2002,1065,2024-01-24 10:48:35,24,17715.40,2,paid
2002,2003,1079,2024-01-18 20:46:40,9,19507.87,1,paid
2003,2004,1014,2024-01-18 23:24:13,26,8786.74,3,cancelled
2004,2005,1056,2024-01-22 23:48:46,8,14404.80,3,cancelled


,order_id,user_id,order_datetime,product_id,price,quantity,status
3000,3001,1100,2024-01-12 20:42:10,4,14665.31,3,paid
3001,3002,1022,2024-01-14 23:16:51,25,2571.62,3,paid
3002,3003,1077,2024-01-19 03:55:19,23,2678.62,2,paid
3003,3004,1049,2024-01-19 05:32:05,3,17080.67,3,paid
3004,3005,1068,2024-01-08 19:57:03,15,10375.87,4,paid


,order_id,user_id,order_datetime,product_id,price,quantity,status
4000,4001,1008,2024-01-10 11:03:20,10,838.11,2,paid
4001,4002,1095,2024-01-21 17:40:56,4,17670.88,2,paid
4002,4003,1053,2024-01-02 11:30:37,7,4581.92,2,paid
4003,4004,1080,2024-01-16 14:14:52,13,824.45,2,paid
4004,4005,1079,2024-01-05 22:06:57,20,3943.42,1,paid


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,5001,1062,2024-02-17 11:39:03,14,16067.85,1,cancelled
1,5002,1078,2024-02-18 15:42:35,30,3716.95,2,cancelled
2,5003,1006,2024-02-21 15:12:54,19,9253.27,1,paid
3,5004,1094,2024-02-04 21:35:01,19,10408.89,4,paid
4,5005,1066,2024-02-20 00:56:21,30,14898.20,2,cancelled


... (6 outputs left out)

,order_id,user_id,order_datetime,product_id,price,quantity,status
2000,12001,1046,2024-03-11 14:00:16,18,8539.25,1,paid
2001,12002,1013,2024-03-09 21:04:11,6,6365.12,2,cancelled
2002,12003,1083,2024-03-06 04:42:08,3,16387.08,4,cancelled
2003,12004,1088,2024-03-27 12:53:59,21,8145.24,4,paid
2004,12005,1090,2024-03-15 01:18:51,5,13733.76,4,paid


,order_id,user_id,order_datetime,product_id,price,quantity,status
3000,13001,1023,2024-03-01 15:18:26,13,9260.34,2,paid
3001,13002,1077,2024-03-04 00:26:18,11,5645.10,4,paid
3002,13003,1007,2024-03-10 08:15:22,14,12358.42,1,paid
3003,13004,1081,2024-03-14 03:04:55,2,17152.54,3,paid
3004,13005,1037,2024-03-24 20:01:54,26,1054.97,4,refunded


,order_id,user_id,order_datetime,product_id,price,quantity,status
4000,14001,1097,2024-03-15 14:44:01,9,4892.68,4,paid
4001,14002,1083,2024-03-26 10:43:54,10,2515.36,3,paid
4002,14003,1100,2024-03-12 20:52:22,27,13882.37,2,paid
4003,14004,1033,2024-03-23 13:23:32,25,16836.92,4,paid
4004,14005,1064,2024-03-11 15:46:43,7,12715.76,1,paid


,file,total,paid,discarded
0,orders_2024_01.csv,5000,4038,962
1,orders_2024_02.csv,5000,3990,1010
2,orders_2024_03.csv,5000,4004,996


## 2. Показатели оплаченных заказов
Выручка позиции равна `price × quantity`. Общая выручка — сумма по оплаченным позициям. Количество заказов считаем по уникальным `order_id`; средний чек — выручка / количество заказов. В этих данных один заказ занимает одну строку. Денежная единица не указана.

In [4]:
paid["line_revenue"] = paid["price"] * paid["quantity"]
revenue = paid["line_revenue"].sum()
orders = paid["order_id"].nunique()
average = revenue / orders if orders else float("nan")
print(f"Общая выручка: {revenue:,.2f}")
print(f"Оплаченных заказов: {orders}")
print(f"Средний чек: {average:,.2f}")

Общая выручка: 307,828,994.03
Оплаченных заказов: 12032
Средний чек: 25,584.19


## 3. Объединение с Excel и средняя цена по категориям
Лист `products` уже содержит названия категорий, поэтому отдельное объединение с листом `categories` не требуется. Соединяем оплаченные позиции с товарами по `product_id`. Объединение `left` сохраняет позиции, а `validate="many_to_one"` проверяет, что ключ товара в справочнике уникален. Дополнительно проверяем, что для каждой позиции найден товар.

В Excel нет цен. Поэтому рассчитываем арифметическое среднее цены `price` по оплаченным строкам каждой категории. Это средняя цена позиции, а не средний чек и не среднее с весами по количеству.

In [5]:
products = pd.read_excel(DATA / "products.xlsx", sheet_name="products", engine="openpyxl")
check_structure(products, "Excel: лист products")
result = paid.merge(products, on="product_id", how="left",
                    validate="many_to_one", indicator=True)
assert result["_merge"].eq("both").all(), "Есть позиции без товара в справочнике"
assert result["category_name"].notna().all(), "Есть товары без категории"
result = result.drop(columns="_merge")
assert len(result) == len(paid), "Изменилось количество позиций"
category_prices = result.groupby("category_name")["price"].mean()
display(category_prices.round(2).to_frame("Средняя цена"))

,product_id,product_name,category_id,category_name,weight_kg
0,1,Product 1,1,Electronics,11.55
1,2,Product 2,3,Sport,2.22
2,3,Product 3,5,Toys,11.43
3,4,Product 4,1,Electronics,4.17
4,5,Product 5,5,Toys,1.81


,Средняя цена
category_name,
Electronics,10157.59
Fashion,10272.67
Home,9874.65
Sport,10300.22
Toys,10141.05


## 4. Сохранение оплаченных позиций
Сохраняем объединённую таблицу в `paid_orders.csv`. `index=False` исключает лишний индекс, `utf-8-sig` позволяет удобно открывать файл в Excel. Повторным чтением проверяем статус, количество строк и выручку.

In [6]:
OUTPUT = Path("paid_orders.csv")
result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
saved = pd.read_csv(OUTPUT)
check_structure(saved, "Повторное чтение paid_orders.csv")
assert len(saved) == len(paid)
assert saved["status"].eq("paid").all()
assert abs(saved["line_revenue"].sum() - revenue) < 0.01
print("CSV сохранён и проверен:", OUTPUT)

,order_id,user_id,order_datetime,product_id,price,quantity,status,line_revenue,product_name,category_id,category_name,weight_kg
0,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid,9172.60,Product 16,3,Sport,1.29
1,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid,18070.52,Product 25,3,Sport,0.20
2,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid,45702.99,Product 9,5,Toys,3.94
3,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid,8268.76,Product 24,1,Electronics,0.56
4,6,1069,2024-01-15 13:09:44,28,19359.94,3,paid,58079.82,Product 28,4,Fashion,10.72


CSV сохранён и проверен: paid_orders.csv


## 5. Оценка фильтрации
Число отброшенных строк равно числу прочитанных минус число оплаченных. Проверяем баланс строк.

In [7]:
total_rows = int(counts["total"].sum())
discarded = int(counts["discarded"].sum())
assert total_rows == len(paid) + discarded
print(f"Прочитано: {total_rows}")
print(f"Оставлено: {len(paid)}")
print(f"Отброшено: {discarded} ({discarded / total_rows:.2%})")

Прочитано: 15000
Оставлено: 12032
Отброшено: 2968 (19.79%)
